In [ ]:
import requests
from bs4 import BeautifulSoup
import gradio as gr
import ollama
import json
import re
from datetime import datetime


In [ ]:
OLLAMA_MODEL = "qwen3:8b"

def ask_ollama(prompt: str, system: str = None) -> str:
    """Small wrapper around Ollama's chat API so every tool calls it the same way."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = ollama.chat(model=OLLAMA_MODEL, messages=messages)
    return response["message"]["content"]

# quick check
print(ask_ollama("Reply with just the word 'ready' if you can read this."))

In [ ]:
def generate_research_questions(topic):
    prompt = f"""
Generate 5 specific and useful research questions or search queries about the following topic:

{topic}

Requirements:
- Each question/query must be on a separate line.
- Do not number them.
- Do not use bullet points.
- Do not add any introduction or explanation.
- Make the questions specific enough to be useful for web research.
"""
    
    response = ask_ollama(prompt)
    
    questions = []
    for line in response.splitlines():
        line = re.sub(r"^\s*[-*•\d.)]+\s*", "", line).strip()
        if line:
            questions.append(line)
    
    return questions

In [ ]:
generate_research_questions("Solar Energy")

In [ ]:
def summarize_source(content):
    prompt = f"""
Summarize the following source in 3-5 clear sentences.

Requirements:
- Use only information explicitly stated in the source.
- Do not add outside information or make unsupported claims.
- Focus on the main ideas, important facts, and findings.
- Keep the summary concise and factual.

Source:
{content}
"""
    
    return ask_ollama(prompt).strip()

In [ ]:
test_content = """
Solar energy is a renewable source of energy that comes from sunlight.
Solar panels use photovoltaic cells to convert sunlight into electricity.
Solar power can reduce dependence on fossil fuels and lower greenhouse gas emissions.
However, solar energy production depends on sunlight and can require significant initial investment.
"""

In [ ]:
summarize_source(test_content)

In [ ]:
def compare_sources(source1, source2):
    prompt = f"""
Compare the following two sources.

Your response must contain exactly two sections:

Agreements:
- List the main points that both sources agree on.

Differences:
- List the main points where the sources disagree, differ in emphasis, or provide different information.

Do not add information that is not present in either source.

Source 1:
{source1}

Source 2:
{source2}
"""
    
    return ask_ollama(prompt).strip()

In [ ]:
source1 = """
Solar energy is a renewable source of energy.
Solar panels can reduce dependence on fossil fuels.
The main challenge is that solar power depends on sunlight.
"""

source2 = """
Solar power is a clean and renewable energy source.
Using solar panels can reduce the use of fossil fuels.
Solar energy production can be affected by weather and the availability of sunlight.
"""

In [ ]:
compare_sources(source1, source2)

In [ ]:
def generate_report(sources, topic):
    sources_text = "\n\n".join(
        f"Source {i + 1}:\n{source}" 
        for i, source in enumerate(sources)
    )

    prompt = f"""
Write a structured research report about the following topic:

{topic}

Use only the information provided in the source summaries below.
Do not invent facts or add information from outside the sources.

The report must contain exactly these sections:

1. Introduction
Briefly introduce the research topic and its context.

2. Key Findings
Present the most important findings from the sources in a clear and organized way.

3. Conclusion
Summarize the main conclusions based on the provided sources.

4. Sources
List the provided sources as Source 1, Source 2, etc.

Source summaries:
{sources_text}
"""
    
    return ask_ollama(prompt).strip()

In [ ]:
test_sources = [
    """
    Solar energy is a renewable source of energy.
    Solar panels convert sunlight into electricity.
    Solar power can reduce dependence on fossil fuels.
    """,

    """
    Solar energy is clean and renewable.
    Advances in photovoltaic technology have improved solar panel efficiency.
    Solar energy production depends on sunlight and weather conditions.
    """,

    """
    Solar power has environmental benefits because it can reduce greenhouse gas emissions.
    However, solar installations can require significant initial investment.
    """
]

In [ ]:
generate_report(test_sources, "Solar Energy")